# Computer Networks · Week 2 — Applications notebook (Google Colab)
### Architecture and performance: layers, reference models and the numbers behind delay

Companion notebook to the **Week 2 teaching pack** (Lecture 2, Seminar 2, Laboratory 2 Linux, Laboratory 2 Windows).

| Part | In the teaching pack | In this notebook |
|---|---|---|
| **A** | Lecture 2 | Build a real Ethernet/IPv4/TCP/HTTP frame byte by byte and demultiplex it; OSI vs TCP/IP; queueing simulations (M/M/1, tail drop, bufferbloat); throughput and the bandwidth–delay product; the end-to-end argument as an experiment; traceroute's mechanism |
| **B** | Seminar 2 | Problems 1–8 in code — including a full decoder for the hexadecimal frame of Problem 2 and simulations that verify Little's law |
| **C** | Laboratory 2 (Linux) | Packet capture on the Colab VM with **tcpdump**, analysis with **tshark** (Wireshark's command-line twin) and **Scapy**; the capture files can be downloaded and opened in Wireshark |
| **D** | Laboratory 2 (Windows) | `tracert` vs `traceroute` seen in the packets, Linux/Windows side by side, files for Wireshark on Windows |

**How to use it.** Run the cells in order. Cells marked ✍️ are for your answers to Q1–Q16, so this notebook can be your report (`File → Download → .ipynb`), together with the capture file `web.pcapng` produced in Task 3.

> **Rules of conduct.** Capture only on your own machine (here: your own Colab VM) and only your own traffic.

## 0 · Setup

In [ ]:
# ---------------------------------------------------------------
#  Helper functions used throughout the notebook. Run this cell first.
# ---------------------------------------------------------------
import os, re, sys, time, math, json, socket, shutil, subprocess, threading, ipaddress
from collections import deque, Counter

IN_COLAB = 'google.colab' in sys.modules
SUDO = '' if os.geteuid() == 0 else 'sudo '

def sh(cmd, timeout=60, show=True):
    """Run a shell command, print its output (stdout+stderr), never raise."""
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        out = (r.stdout or '') + (r.stderr or '')
    except subprocess.TimeoutExpired as e:
        o = e.stdout or ''
        out = (o.decode(errors='replace') if isinstance(o, bytes) else o) + f'\n[stopped after {timeout} s]'
    if show:
        print(f'$ {cmd}\n{out.rstrip()}\n')
    return out

def have(tool):
    return shutil.which(tool) is not None

def default_route():
    """(interface, gateway) used to reach the Internet."""
    out = sh('ip route get 1.1.1.1', show=False)
    dev = re.search(r'\bdev (\S+)', out)
    via = re.search(r'\bvia (\S+)', out)
    src = re.search(r'\bsrc (\S+)', out)
    return (dev.group(1) if dev else 'any', via.group(1) if via else None, src.group(1) if src else None)

def resolve(host):
    try:
        return socket.gethostbyname(host)
    except Exception as e:
        print(f'cannot resolve {host}: {e}')
        return None

def tcp_rtt(host, port=443, n=5, timeout=3):
    """Measure RTT as the time a TCP connect() takes (one SYN / SYN-ACK exchange).
    Works where ICMP ping is blocked (as on many cloud VMs)."""
    ip = resolve(host)
    if not ip:
        return []
    rtts = []
    for _ in range(n):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        t0 = time.perf_counter()
        try:
            s.connect((ip, port))
            rtts.append((time.perf_counter() - t0) * 1000)
        except Exception as e:
            print(f'  connect to {ip}:{port} failed: {e}')
        finally:
            s.close()
        time.sleep(0.2)
    if rtts:
        print(f'TCP-connect RTT to {host} ({ip}:{port}): min/avg/max = '
              f'{min(rtts):.2f}/{sum(rtts)/len(rtts):.2f}/{max(rtts):.2f} ms  ({len(rtts)}/{n} answered)')
    return rtts

def fmt_time(seconds):
    """Pretty-print a duration."""
    s = abs(seconds)
    if s < 1e-6: return f'{seconds*1e9:.3g} ns'
    if s < 1e-3: return f'{seconds*1e6:.4g} µs'
    if s < 1:    return f'{seconds*1e3:.4g} ms'
    if s < 120:  return f'{seconds:.4g} s'
    if s < 7200: return f'{seconds/60:.4g} min'
    if s < 2*86400: return f'{seconds/3600:.4g} h'
    return f'{seconds/86400:.4g} days'

def check(name, got, expected, rel=0.02):
    ok = math.isclose(got, expected, rel_tol=rel, abs_tol=1e-12)
    print(f"{'✅' if ok else '❌'} {name}: computed {got:.6g}, expected ≈ {expected:.6g}")
    return ok

print('Helpers loaded. Running in Colab:', IN_COLAB, '| root:', os.geteuid() == 0)

In [ ]:
# Command-line tools (tcpdump, tshark, traceroute, dig, curl) and Scapy. About 1 minute on Colab.
sh(f"echo 'wireshark-common wireshark-common/install-setuid boolean false' | {SUDO}debconf-set-selections", show=False)
sh(f'{SUDO}apt-get -qq update > /dev/null 2>&1; DEBIAN_FRONTEND=noninteractive {SUDO}apt-get -qq install -y '
   'tcpdump tshark traceroute dnsutils curl iproute2 iputils-ping > /dev/null 2>&1; echo apt done', timeout=900)
sh(f'{sys.executable} -m pip -q install scapy 2>&1 | tail -1', timeout=300)
for t in ['tcpdump', 'tshark', 'editcap', 'traceroute', 'dig', 'curl']:
    print(f'{t:11s}', '✅' if have(t) else '❌ missing')

In [ ]:
import struct, zlib, hashlib, random, signal, shlex
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.grid': True, 'grid.alpha': .3})
K, M, G = 1e3, 1e6, 1e9

def hexdump(data, width=16):
    for off in range(0, len(data), width):
        chunk = data[off:off + width]
        hx = ' '.join(f'{b:02x}' for b in chunk)
        asc = ''.join(chr(b) if 32 <= b < 127 else '.' for b in chunk)
        print(f'{off:04x}  {hx:<{width*3}} {asc}')

---
# Part A · Lecture 2 — the lecture in code

## A.1 Service, interface, protocol
The **service** of TCP is a reliable byte stream; its **interface** is the socket calls; its **protocol** is the segments exchanged with the peer. Below, two processes on the same machine use only the interface — they never see a segment.

In [ ]:
srv = socket.socket(); srv.bind(('127.0.0.1', 0)); srv.listen(1)
port = srv.getsockname()[1]

def echo_server():
    conn, _ = srv.accept()                     # interface: accept
    data = conn.recv(1024)                     # interface: recv
    conn.sendall(data.upper())                 # interface: send
    conn.close()

threading.Thread(target=echo_server, daemon=True).start()
c = socket.create_connection(('127.0.0.1', port))   # interface: connect (TCP's 3-way handshake happens underneath)
c.sendall(b'hello, transport layer')
print('reply:', c.recv(1024)); c.close(); srv.close()

## A.2 Encapsulation, byte by byte
We build the frame of **Lecture 2, section 3.1**: a 400-byte HTTP request inside TCP (20 B), IPv4 (20 B) and Ethernet (14 B header + 4 B trailer). Every checksum is computed exactly as real hosts do.

In [ ]:
def inet_checksum(data: bytes) -> int:
    """The Internet checksum (RFC 1071): one's-complement sum of 16-bit words."""
    if len(data) % 2:
        data += b'\0'
    s = sum(struct.unpack(f'!{len(data)//2}H', data))
    while s >> 16:
        s = (s & 0xFFFF) + (s >> 16)
    return ~s & 0xFFFF

def mac(s): return bytes.fromhex(s.replace(':', ''))
def ip(s):  return socket.inet_aton(s)

def tcp_segment(src_ip, dst_ip, sport, dport, payload=b'', seq=1, ack=1, flags=0x18, window=64240):
    hdr = struct.pack('!HHIIBBHHH', sport, dport, seq, ack, 5 << 4, flags, window, 0, 0)
    pseudo = struct.pack('!4s4sBBH', ip(src_ip), ip(dst_ip), 0, 6, len(hdr) + len(payload))
    csum = inet_checksum(pseudo + hdr + payload)
    return hdr[:16] + struct.pack('!H', csum) + hdr[18:] + payload

def udp_datagram(src_ip, dst_ip, sport, dport, payload):
    length = 8 + len(payload)
    hdr = struct.pack('!HHHH', sport, dport, length, 0)
    pseudo = struct.pack('!4s4sBBH', ip(src_ip), ip(dst_ip), 0, 17, length)
    return struct.pack('!HHHH', sport, dport, length, inet_checksum(pseudo + hdr + payload)) + payload

def ipv4_packet(src_ip, dst_ip, proto, payload, ttl=64, ident=0x1c46):
    hdr = struct.pack('!BBHHHBBH4s4s', 0x45, 0, 20 + len(payload), ident, 0x4000, ttl, proto, 0, ip(src_ip), ip(dst_ip))
    return hdr[:10] + struct.pack('!H', inet_checksum(hdr)) + hdr[12:] + payload

def ethernet_frame(dst_mac, src_mac, ethertype, payload, with_fcs=True):
    frame = mac(dst_mac) + mac(src_mac) + struct.pack('!H', ethertype) + payload
    return frame + struct.pack('<I', zlib.crc32(frame)) if with_fcs else frame

# A 400-byte HTTP request
lines = ['GET /index.html HTTP/1.1', 'Host: www.example.org', 'User-Agent: course-notebook/1.0', 'Accept: */*']
base = '\r\n'.join(lines) + '\r\n'
pad = 400 - len(base) - len('X-Padding: \r\n\r\n')
http = (base + 'X-Padding: ' + 'x' * pad + '\r\n\r\n').encode()

CLIENT_MAC, GW_MAC = '52:54:00:3a:7f:10', '52:54:00:12:34:01'
CLIENT_IP, SERVER_IP = '10.20.3.57', '93.184.215.14'
seg = tcp_segment(CLIENT_IP, SERVER_IP, 51234, 80, http)
pkt = ipv4_packet(CLIENT_IP, SERVER_IP, 6, seg)
frame = ethernet_frame(GW_MAC, CLIENT_MAC, 0x0800, pkt)

for name, size in [('HTTP request', len(http)), ('+ TCP header', len(seg)), ('+ IPv4 header', len(pkt)), ('+ Ethernet header and trailer', len(frame))]:
    print(f'{name:32s} {size:4d} bytes')
print(f'Headers: {len(frame) - len(http)} of {len(frame)} bytes\n')
hexdump(frame[:80]); print('...')

In [ ]:
# The same frame decoded by an independent implementation (Scapy), to check our bytes and checksums
from scapy.all import Ether, IP as sIP, TCP as sTCP
f = Ether(frame[:-4])                       # capture tools usually omit the 4-byte FCS
print(f.summary())
g = f.copy(); del g[sIP].chksum; del g[sTCP].chksum; g = Ether(bytes(g))   # let Scapy recompute them
print(f'IP  checksum: ours 0x{f[sIP].chksum:04x}   Scapy 0x{g[sIP].chksum:04x}')
print(f'TCP checksum: ours 0x{f[sTCP].chksum:04x}   Scapy 0x{g[sTCP].chksum:04x}')
print('FCS (CRC-32) :', frame[-4:].hex())
print('MTU check: max TCP payload over Ethernet =', 1500 - 20 - 20, 'bytes')

### Demultiplexing
At each layer the receiver reads **one field** to decide who gets the payload: EtherType → IP Protocol → destination port.

In [ ]:
ETHERTYPE = {0x0800: 'IPv4', 0x86DD: 'IPv6', 0x0806: 'ARP'}
IPPROTO = {1: 'ICMP', 6: 'TCP', 17: 'UDP'}
PORTS = {53: 'DNS', 80: 'HTTP', 443: 'HTTPS', 22: 'SSH', 25: 'SMTP'}

def port_class(p):
    return 'well-known' if p < 1024 else 'registered' if p < 49152 else 'dynamic'

def demultiplex(frame):
    et = struct.unpack('!H', frame[12:14])[0]
    print(f'Link      reads EtherType 0x{et:04x}  → {ETHERTYPE.get(et, "?")}')
    if et != 0x0800: return
    ihl = (frame[14] & 0x0F) * 4
    proto = frame[14 + 9]
    print(f'Network   reads Protocol  {proto:6d}  → {IPPROTO.get(proto, "?")}')
    l4 = frame[14 + ihl:]
    sport, dport = struct.unpack('!HH', l4[:4])
    print(f'Transport reads dst port  {dport:6d}  → {PORTS.get(dport, "?")}  '
          f'(source port {sport} is {port_class(sport)}, destination port {dport} is {port_class(dport)})')
demultiplex(frame)

### Which addresses change at each hop?
The destination **IP** address is the server's all the way; the destination **MAC** address is only the next hop. Each router builds a new frame and decreases the TTL.

In [ ]:
path = [  # (node, MAC of the interface on the left, MAC on the right)
    ('client', None, '52:54:00:3a:7f:10'),
    ('R1 (gateway)', '52:54:00:12:34:01', 'aa:00:00:00:01:02'),
    ('R2', 'aa:00:00:00:02:01', 'aa:00:00:00:02:02'),
    ('server', 'bb:00:00:00:00:80', None),
]
ttl = 64
print(f'{"link":24s} {"src MAC":18s} {"dst MAC":18s} {"src IP":12s} {"dst IP":14s} TTL')
for (a, _, a_out), (z, z_in, _) in zip(path, path[1:]):
    print(f'{a + " → " + z:24s} {a_out:18s} {z_in:18s} {CLIENT_IP:12s} {SERVER_IP:14s} {ttl}')
    ttl -= 1

## A.3 Reference models

In [ ]:
OSI = [(7, 'Application', 'HTTP, SMTP, DNS'), (6, 'Presentation', 'UTF-8, JSON, TLS — inside applications/libraries'),
       (5, 'Session', 'HTTP sessions, TLS resumption — inside applications'), (4, 'Transport', 'TCP, UDP, QUIC'),
       (3, 'Network', 'IP'), (2, 'Data link', 'Ethernet, Wi-Fi'), (1, 'Physical', 'copper, fibre, radio signalling')]
FIVE = {7: 'Application', 6: 'Application', 5: 'Application', 4: 'Transport', 3: 'Network', 2: 'Link', 1: 'Physical'}
RFC1122 = {7: 'Application', 6: 'Application', 5: 'Application', 4: 'Transport', 3: 'Internet', 2: 'Link', 1: 'Link'}
print(f'{"#":2s} {"OSI (7)":13s} {"5-layer":12s} {"RFC 1122 (4)":13s} where it lives in the Internet')
for n, name, where in OSI:
    print(f'{n:<2d} {name:13s} {FIVE[n]:12s} {RFC1122[n]:13s} {where}')
print('\nMnemonic, layer 1 up: "Please Do Not Throw Sausage Pizza Away"')

In [ ]:
# The narrow waist (Figure 2)
tops = ['HTTP', 'SMTP', 'DNS', 'SSH', 'RTP', 'QUIC apps']; mids = ['TCP', 'UDP', 'QUIC']
bots = ['Ethernet', 'Wi-Fi', '4G/5G', 'DSL', 'fibre', 'satellite']
fig, ax = plt.subplots(figsize=(8, 4.5)); ax.axis('off')
def row(items, y, w, color):
    xs = np.linspace(-w, w, len(items))
    for x, t in zip(xs, items):
        ax.text(x, y, t, ha='center', va='center', bbox=dict(boxstyle='round', fc=color), fontsize=9)
row(tops, 4, 3.3, '#ffe0b3'); row(mids, 3, 1.6, '#cfe3ff'); row(['IP'], 2, 0, '#ffb3b3'); row(bots, 1, 3.3, '#d5f5d5')
ax.plot([-3.8, -0.4, -3.8], [4.4, 2, 0.6], 'k--', lw=.8); ax.plot([3.8, 0.4, 3.8], [4.4, 2, 0.6], 'k--', lw=.8)
ax.set_xlim(-4, 4); ax.set_ylim(0.4, 4.6); ax.set_title('The hourglass: one network-layer protocol in the middle'); plt.show()

## A.4 Who makes the rules
IETF (Internet-Drafts → RFCs: *Proposed Standard* → *Internet Standard*), IEEE 802 (Ethernet, Wi-Fi), ITU-T and 3GPP (telecom, 4G/5G), W3C and WHATWG (Web), IANA and the RIRs (numbers; Romania: RIPE NCC). *"We reject kings, presidents and voting. We believe in rough consensus and running code."* (D. Clark, 1992)

## A.5 Queueing delay in depth
Traffic intensity `ρ = L·a / R`. In the M/M/1 model the average time at the router is `d = (L/R) / (1 − ρ)`. We check that formula with a **simulation** of random arrivals and random packet lengths (Lindley's recursion: each packet waits for what the previous one left behind).

In [ ]:
def mm1_simulate(rho, n=200_000, service_mean=1.0, seed=0):
    """Return (waiting times, sojourn times, arrival times) for an M/M/1 queue."""
    rng = np.random.default_rng(seed)
    inter = rng.exponential(service_mean / rho, n)
    serv = rng.exponential(service_mean, n)
    arrivals = np.cumsum(inter)
    wait = np.zeros(n)
    for i in range(1, n):
        wait[i] = max(0.0, wait[i-1] + serv[i-1] - inter[i])
    return wait, wait + serv, arrivals

rhos = [0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95]
sim = [mm1_simulate(r)[1].mean() for r in rhos]
grid = np.linspace(0, 0.97, 300)
plt.plot(grid, 1 / (1 - grid), label='M/M/1 formula  (L/R)/(1−ρ)')
plt.plot(rhos, sim, 'o', label='simulation (200 000 packets)')
plt.ylim(0, 25); plt.xlabel('traffic intensity ρ'); plt.ylabel('average delay (in units of L/R)')
plt.title('Delay explodes as ρ → 1'); plt.legend(); plt.show()
for r, s in zip(rhos, sim):
    print(f'ρ = {r:4.2f}: formula {1/(1-r):6.2f} · L/R   simulation {s:6.2f} · L/R')

### Loss with finite buffers, and bufferbloat
A buffer of **K** packets drops arrivals when full (*tail drop*). The second plot shows why huge buffers are not the answer: during overload a full buffer is simply a long queue that every packet must wait behind.

In [ ]:
def finite_queue(rho, K, n=100_000, seed=1, service_mean=1.0):
    """M/M/1/K by event simulation. Returns (loss fraction, mean delay of delivered packets)."""
    rng = np.random.default_rng(seed)
    t, departures, lost, delays = 0.0, deque(), 0, []
    for _ in range(n):
        t += rng.exponential(service_mean / rho)
        while departures and departures[0] <= t:
            departures.popleft()
        if len(departures) >= K + 1:          # K waiting + 1 in transmission
            lost += 1; continue
        start = departures[-1] if departures else t
        dep = max(start, t) + rng.exponential(service_mean)
        departures.append(dep); delays.append(dep - t)
    return lost / n, float(np.mean(delays))

rho_axis = np.linspace(0.5, 1.3, 9)
for K in (5, 20, 100):
    plt.plot(rho_axis, [finite_queue(r, K, n=40_000)[0] for r in rho_axis], 'o-', label=f'buffer K = {K}')
plt.xlabel('traffic intensity ρ'); plt.ylabel('fraction of packets lost'); plt.legend()
plt.title('Tail-drop loss: above ρ = 1 no buffer size helps'); plt.show()

In [ ]:
Ks = [5, 10, 20, 50, 100, 200, 500, 1000]
tx_ms = 1500 * 8 / 10e6 * 1e3            # one 1500-byte packet on a 10 Mbit/s home uplink = 1.2 ms
res = [finite_queue(1.05, K, n=60_000) for K in Ks]
plt.semilogx(Ks, [d * tx_ms for _, d in res], 'o-')
plt.xlabel('buffer size (packets)'); plt.ylabel('average delay (ms)')
plt.title('Bufferbloat: one upload at ρ = 1.05 on a 10 Mbit/s uplink'); plt.show()
for K, (loss, d) in zip(Ks, res):
    print(f'K = {K:5d}: loss {loss:5.1%}  delay {d*tx_ms:7.1f} ms  ← added to every packet, including your video call')

## A.6 Throughput and the bandwidth–delay product
`throughput = min(R₁, …, Rₙ)` (or the fair share of a shared link) and `BDP = R × RTT` — the data that must be in flight to keep the path full. A sender allowed only **W** bytes outstanding reaches at most `W / RTT`.

In [ ]:
def bdp_bytes(R_bps, rtt_s): return R_bps * rtt_s / 8
def window_limited(W_bytes, rtt_s, R_bps): return min(R_bps, W_bytes * 8 / rtt_s)

b = bdp_bytes(1*G, 0.030)
print(f'BDP of 1 Gbit/s × 30 ms = {b*8/1e6:.0f} Mbit = {b/1e6:.2f} MB')
print(f'With a 64 KB window: {window_limited(65535, 0.030, 1*G)/1e6:.1f} Mbit/s on a 1 Gbit/s path')

rtts = np.linspace(0.001, 0.3, 300)
for W in (65535, 1<<20, 16<<20):
    plt.plot(rtts * 1e3, [window_limited(W, r, 1*G) / 1e6 for r in rtts], label=f'window {W/1024:.0f} KB')
plt.xlabel('RTT (ms)'); plt.ylabel('max throughput (Mbit/s)'); plt.yscale('log'); plt.legend()
plt.title('A 1 Gbit/s path: the window, not the link, limits long-distance transfers'); plt.show()

### The opening question
You: links 1 Gbit/s, 10 Gbit/s, 100 Mbit/s, RTT 10 ms. Neighbour: 1 Gbit/s everywhere, RTT 300 ms. A simple model of a TCP download: one RTT for the handshake, then a window that starts at 10 segments and doubles every RTT (*slow start*) until it fills the path.

In [ ]:
def download_time(size_bytes, R_bps, rtt, mss=1460, init_segments=10):
    t, sent, cwnd = rtt, 0.0, init_segments * mss        # handshake first
    full = bdp_bytes(R_bps, rtt)
    while True:
        per_rtt = min(cwnd, full)
        if size_bytes - sent <= per_rtt:
            return t + rtt / 2 + (size_bytes - sent) * 8 / R_bps
        sent += per_rtt; t += rtt; cwnd *= 2

sizes = np.logspace(4, 10.5, 200)
you = [download_time(s, 100*M, 0.010) for s in sizes]
nb_ = [download_time(s, 1*G, 0.300) for s in sizes]
plt.loglog(sizes / 1e6, you, label='you: 100 Mbit/s bottleneck, RTT 10 ms')
plt.loglog(sizes / 1e6, nb_, label='neighbour: 1 Gbit/s, RTT 300 ms')
plt.xlabel('file size (MB)'); plt.ylabel('download time (s)'); plt.legend()
plt.title('Small files: delay decides · large files: throughput decides'); plt.show()
for s in (1e6, 10e9):
    print(f'{s/1e6:>8.0f} MB: you {download_time(s, 100*M, .01):8.2f} s   neighbour {download_time(s, 1*G, .3):8.2f} s')

## A.7 The end-to-end argument, as an experiment
A file crosses three hops. Every link checks a CRC and retransmits corrupted frames, so each hop is "reliable". But a router occasionally corrupts a byte **in its memory**, after the link check. Only a checksum computed by the sender over the whole file and verified by the receiver catches this.

In [ ]:
def transfer(file_bytes, hops=3, link_ber=1e-5, router_bug=0.002, frame=1000, rng=None):
    rng = rng or random.Random(0)
    frames = [bytearray(file_bytes[i:i+frame]) for i in range(0, len(file_bytes), frame)]
    link_retx = 0
    for hop in range(hops):
        for k, f in enumerate(frames):
            while True:                                       # hop-by-hop reliability: CRC + retransmit
                sent_crc = zlib.crc32(f)
                rx = bytearray(f)
                if rng.random() < link_ber * len(f) * 8:      # bit error on the wire
                    rx[rng.randrange(len(rx))] ^= 1 << rng.randrange(8)
                if zlib.crc32(rx) == sent_crc: break
                link_retx += 1
            if hop < hops - 1 and rng.random() < router_bug:  # corruption inside the router, after the check
                rx[rng.randrange(len(rx))] ^= 0xFF
            frames[k] = rx
    return bytes(b''.join(frames)), link_retx

rng = random.Random(42)
data = os.urandom(1_000_000)
sender_digest = hashlib.sha256(data).hexdigest()
bad = 0
for i in range(10):
    received, retx = transfer(data, rng=rng)
    ok = hashlib.sha256(received).hexdigest() == sender_digest
    bad += not ok
    print(f'transfer {i+1:2d}: {retx:3d} link retransmissions, every link check passed → end-to-end check: {"OK" if ok else "CORRUPTED"}')
print(f'\n{bad}/10 files were corrupted even though every hop was "reliable".')
print('Fix: the receiver compares SHA-256 with the sender\'s and asks for the file again (end-to-end retry).')

## A.8 How traceroute works
Probes with TTL = 1, 2, 3, … Each router decreases the TTL and, when it reaches 0, discards the probe and returns **ICMP Time Exceeded (type 11)**. The destination answers **Port Unreachable** (type 3, code 3 — Linux UDP probes) or **Echo Reply** (type 0 — Windows `tracert`).

In [ ]:
def traceroute_sim(routers, dest, probes_per_hop=3, style='linux'):
    sent = 0
    for ttl in range(1, len(routers) + 2):
        sent += probes_per_hop
        t = ttl
        for node in routers:                 # each router decrements the TTL
            t -= 1
            if t == 0:
                print(f'TTL={ttl:2d}: expires at {node:12s} → ICMP Time Exceeded (type 11)'); break
        else:
            reply = 'ICMP Port Unreachable (type 3)' if style == 'linux' else 'ICMP Echo Reply (type 0)'
            print(f'TTL={ttl:2d}: reaches   {dest:12s} → {reply}'); break
    return sent

n = traceroute_sim(['gw', 'core1', 'nren-1', 'nren-2', 'transit-1'], 'server')
print('probes sent:', n)
print('\nExercise 5: 14 hops × 3 probes =', 14 * 3, 'probes; hops 1–13 send Time Exceeded, hop 14 Port Unreachable / Echo Reply')

## A.9 End-of-lecture exercises, checked by code

In [ ]:
print('1. 20-byte message over UDP (8) + IPv6 (40) + Ethernet (18)')
check('   application data fraction', 20 / (20 + 8 + 40 + 18), 0.23, rel=0.02)
print('2. 1000-byte packets, 8 Mbit/s link')
LR = 8000 / 8e6
check('   arrival rate for ρ = 1 (packets/s)', 1 / LR, 1000)
for a, exp in [(500, 2e-3), (800, 5e-3), (950, 20e-3)]:
    rho = a * LR
    check(f'   delay at {a} pkt/s (ρ = {rho:.2f}) (s)', LR / (1 - rho), exp)
print('3. 100 / 40 / 1000 Mbit/s, the last shared by 50 transfers')
check('   throughput (Mbit/s)', min(100, 40, 1000 / 50), 20)
print('4. BDP of 10 Gbit/s × 80 ms')
check('   MB', bdp_bytes(10*G, 0.080) / 1e6, 100)
print('5. traceroute, 14 hops, 3 probes per hop')
check('   probes', 14 * 3, 42)

**Review questions**: (1) service/interface/protocol at another layer · (2) which protocols the router of Figure 1 takes part in · (3) the three fields read to deliver a frame to a browser · (4) why the MAC changes at every hop and the IP does not · (5) OSI layers without a TCP/IP counterpart · (6) narrow waist: easy to add apps, hard to change IP · (7) "60 % used, 40 % spare, no need to upgrade?" · (8) why a bigger buffer is not a cure for loss · (9) the end-to-end argument in one sentence.

**✍️ Your answers** (double-click to edit)

- **R1:** 
- **R2:** 
- **R3:** 
- **R4:** 
- **R5:** 
- **R6:** 
- **R7:** 
- **R8:** 
- **R9:** 

---
# Part B · Seminar 2 — Headers, queues and throughput

## B.1 Problem 1 — which layer?
Type your answers in `MINE`, then run the cell.

In [ ]:
FUNCTIONS = {
    'a': 'Choosing the outgoing link for a packet addressed to another country',
    'b': 'Converting a document from one character encoding to another',
    'c': 'Retransmitting a segment whose acknowledgement did not arrive',
    'd': 'Representing a 1 as a voltage transition on a cable',
    'e': 'Detecting that a frame was corrupted on a Wi-Fi link',
    'f': 'Resuming a long file download from where it stopped after a disconnection',
    'g': 'Delivering an incoming segment to the right process on a host',
    'h': 'Deciding which of several hosts on a shared cable may transmit now',
}
SOLUTION = {'a': ('network', 3), 'b': ('application', 6), 'c': ('transport', 4), 'd': ('physical', 1),
            'e': ('link', 2), 'f': ('application', 5), 'g': ('transport', 4), 'h': ('link', 2)}
MINE = {'a': ('network', 3), 'b': ('', 0), 'c': ('', 0), 'd': ('', 0), 'e': ('', 0), 'f': ('', 0), 'g': ('', 0), 'h': ('', 0)}

score = 0
for k, text in FUNCTIONS.items():
    ok = MINE[k] == SOLUTION[k]; score += ok
    print(f"{'✅' if ok else '❌'} ({k}) {text[:62]:62s} → 5-layer: {SOLUTION[k][0]:11s} OSI: {SOLUTION[k][1]}")
print(f'\nScore: {score}/8')

## B.2 Problem 2 — decoding a frame by hand (and by code)
The decoder below reads each field exactly as the seminar's layout describes. Try it on paper first.

In [ ]:
HEX = """
0000  52 54 00 12 34 01 52 54 00 3a 7f 10 08 00 45 00
0010  00 3d 1c 46 40 00 40 11 06 d5 0a 14 03 39 0a 14
0020  00 35 c8 22 00 35 00 29 96 d2 3b 7a 01 00 00 01
0030  00 00 00 00 00 00 03 77 77 77 07 65 78 61 6d 70
0040  6c 65 03 6f 72 67 00 00 01 00 01
"""
def from_dump(text):
    return bytes.fromhex(''.join(' '.join(l.split()[1:]) for l in text.strip().splitlines()))

def fmt_mac(b): return ':'.join(f'{x:02x}' for x in b)

def decode_ethernet(f):
    d = {'dst_mac': fmt_mac(f[0:6]), 'src_mac': fmt_mac(f[6:12]), 'ethertype': struct.unpack('!H', f[12:14])[0]}
    return d, f[14:]

def decode_ipv4(p):
    v_ihl, tos, total, ident, flags_frag, ttl, proto, csum = struct.unpack('!BBHHHBBH', p[:12])
    ihl = (v_ihl & 0x0F) * 4
    d = {'version': v_ihl >> 4, 'header_len': ihl, 'total_len': total, 'id': ident,
         'DF': bool(flags_frag & 0x4000), 'ttl': ttl, 'protocol': proto, 'checksum': csum,
         'src': socket.inet_ntoa(p[12:16]), 'dst': socket.inet_ntoa(p[16:20]),
         'checksum_ok': inet_checksum(p[:ihl]) == 0}       # summing a correct header gives 0
    return d, p[ihl:total]

def decode_udp(u):
    sport, dport, length, csum = struct.unpack('!HHHH', u[:8])
    return {'sport': sport, 'dport': dport, 'length': length, 'checksum': csum}, u[8:length]

def decode_tcp(t):
    sport, dport, seq, ack, off, flags, win, csum, urg = struct.unpack('!HHIIBBHHH', t[:20])
    hl = (off >> 4) * 4
    names = [n for bit, n in [(0x02, 'SYN'), (0x10, 'ACK'), (0x01, 'FIN'), (0x04, 'RST'), (0x08, 'PSH')] if flags & bit]
    return {'sport': sport, 'dport': dport, 'seq': seq, 'ack': ack, 'header_len': hl,
            'flags': '+'.join(names), 'window': win, 'options_len': hl - 20}, t[hl:]

def dns_qname(msg, off=12):
    labels = []
    while msg[off]:
        n = msg[off]; labels.append(msg[off+1:off+1+n].decode()); off += 1 + n
    qtype, qclass = struct.unpack('!HH', msg[off+1:off+5])
    return '.'.join(labels), qtype, qclass

frame2 = from_dump(HEX)
eth, rest = decode_ethernet(frame2)
ip4, rest = decode_ipv4(rest)
udp, dns = decode_udp(rest)
print('1.', f"dst MAC {eth['dst_mac']}  src MAC {eth['src_mac']}  EtherType 0x{eth['ethertype']:04x} → {ETHERTYPE[eth['ethertype']]}")
print('2.', f"IP version {ip4['version']}, header {ip4['header_len']} B, total length {ip4['total_len']} B; "
      f"frame shown = {len(frame2)} B = 14 + {len(frame2) - 14}  → consistent: {len(frame2) - 14 == ip4['total_len']}")
print('3.', f"TTL {ip4['ttl']}, protocol {ip4['protocol']} = {IPPROTO[ip4['protocol']]}, {ip4['src']} → {ip4['dst']}  "
      f"(header checksum 0x{ip4['checksum']:04x} valid: {ip4['checksum_ok']})")
print('4.', f"ports {udp['sport']} → {udp['dport']} ({port_class(udp['sport'])} → {PORTS[udp['dport']]}); "
      f"UDP length {udp['length']} = 8 + {len(dns)} bytes of DNS")
name, qtype, qclass = dns_qname(dns)
print('5.', f"name looked up: {name}   type {qtype} (A)  class {qclass} (IN)")
same = ipaddress.ip_network(ip4['src'] + '/24', strict=False) == ipaddress.ip_network(ip4['dst'] + '/24', strict=False)
print('6.', f"{ip4['src']}/24 and {ip4['dst']}/24 in the same subnet? {same} → the frame goes to the "
      f"{'DNS server directly' if same else 'default gateway, so the destination MAC is the gateway’s'}")

## B.3 Problem 3 — how much of the link is useful?
On the wire: preamble 8 B + header 14 B + payload + FCS 4 B + inter-frame gap 12 B. Payloads below 46 B are padded to 46.

In [ ]:
def wire_bytes(payload):
    return 8 + 14 + max(payload, 46) + 4 + 12

check('1. byte-times of a full frame', wire_bytes(1500), 1538)
check('2. goodput TCP/IPv4 on 100 Mbit/s (Mbit/s)', 1460 / wire_bytes(1500) * 100, 94.9, rel=0.002)
check('3. goodput TCP/IPv6 (Mbit/s)', 1440 / wire_bytes(1500) * 100, 93.6, rel=0.002)
check('4. audio fraction, 20-byte samples over UDP/IPv4', 20 / wire_bytes(20 + 8 + 20), 20 / 86)

app = np.arange(1, 1473)
plt.plot(app, app / np.array([wire_bytes(a + 28) for a in app]) * 100)
plt.xlabel('application bytes per UDP/IPv4 packet'); plt.ylabel('useful share of a 100 Mbit/s link (%)')
plt.title('Small messages waste the link'); plt.show()

## B.4 Problem 4 — traffic intensity (10 Mbit/s link, 1250-byte packets, M/M/1)

In [ ]:
LR = 1250 * 8 / 10e6
check('1. L/R (s)', LR, 1e-3); check('1. arrival rate for ρ = 1', 1 / LR, 1000)
print('\n   a (pkt/s)    ρ     delay')
for a in (500, 800, 900, 950, 990):
    rho = a * LR
    print(f'   {a:8d}   {rho:5.2f}   {LR / (1 - rho) * 1e3:6.0f} ms')
months_double = math.log(0.9 / 0.8) / math.log(1.05)
months_ten = math.log(0.98 / 0.8) / math.log(1.05)
check('3. months until the delay doubles', months_double, 2.4, rel=0.03)
check('3. months until the delay is ten times', months_ten, 4.2, rel=0.03)

## B.5 Problem 5 — Little's law `N = λ × T`
We also **verify** it on a simulated queue: the time-average number of packets inside, measured independently, equals λ × T.

In [ ]:
check('1. time in buffer (s)', 40 / 2000, 0.020)
check('2. requests in progress', 300 * 0.05, 15)
check('3. throughput with 64 KB in flight, RTT 20 ms (Mbit/s)', 65536 * 8 / 0.02 / 1e6, 26.2, rel=0.01)

wait, sojourn, arr = mm1_simulate(0.8, n=100_000, service_mean=1e-3)
deps = arr + sojourn
events = sorted([(t, +1) for t in arr] + [(t, -1) for t in deps])
area, n_in, last = 0.0, 0, events[0][0]
for t, step in events:
    area += n_in * (t - last); n_in += step; last = t
N_measured = area / (deps.max() - arr[0])
lam = len(arr) / (arr[-1] - arr[0])
print(f'\nSimulation (ρ = 0.8): N measured = {N_measured:.3f}   λ × T = {lam:.0f}/s × {sojourn.mean()*1e3:.3f} ms = {lam * sojourn.mean():.3f}')

## B.6 Problem 6 — shared core (servers 20 Mbit/s, clients 8 Mbit/s, 10 transfers share a core link R)

In [ ]:
def flow_throughput(R_core, n=10, server=20*M, client=8*M):
    return min(server, client, R_core / n)
check('1. R = 1 Gbit/s (Mbit/s)', flow_throughput(1*G) / M, 8)
check('2. R = 50 Mbit/s (Mbit/s)', flow_throughput(50*M) / M, 5)
print('3. the core is the bottleneck when R/10 < 8 Mbit/s, i.e. R <', 8 * 10, 'Mbit/s')
Rs = np.linspace(10, 200, 400) * M
plt.plot(Rs / M, [flow_throughput(r) / M for r in Rs]); plt.axvline(80, ls=':', color='r')
plt.xlabel('core link rate R (Mbit/s)'); plt.ylabel('throughput per transfer (Mbit/s)'); plt.show()

## B.7 Problem 7 — the bandwidth–delay product

In [ ]:
for name, R_, rtt, exp in [('(a) campus LAN', 1*G, 0.0005, 62.5e3), ('(b) Bucharest–Frankfurt', 1*G, 0.030, 3.75e6),
                           ('(c) geostationary satellite', 50*M, 0.600, 3.75e6)]:
    check(f'1. {name} BDP (bytes)', bdp_bytes(R_, rtt), exp)
check('2. 64 KB window on path (b) (Mbit/s)', window_limited(65535, 0.030, 1*G) / M, 17.5, rel=0.01)
print('   = {:.1%} of the link'.format(window_limited(65535, 0.030, 1*G) / G))
print('3. window needed to fill (b): at least one BDP = 3.75 MB → TCP window scale option (RFC 7323)')

## B.8 Problem 8 — the end-to-end argument (discussion)
1. Routers check every TCP checksum so hosts need not. 2. A radio link retransmits lost frames locally. 3. An ISP proxy recompresses images. 4. Banks use TLS end to end over an already link-encrypted network.

<details><summary>Show the seminar's answers</summary>

1. **Inconsistent** as a replacement — corruption can happen inside routers and hosts; only an end-to-end check catches it (A.7 above shows this).
2. **Consistent** — a performance enhancement; TCP still checks end to end.
3. **Problematic** — the network changes data the endpoints did not ask it to change; defeated by HTTPS.
4. **Consistent** — only the endpoints can guarantee confidentiality between them.
</details>

**✍️ Your answers** (double-click to edit)

- **P8.1:** 
- **P8.2:** 
- **P8.3:** 
- **P8.4:** 

**Homework**: (1) lecture exercises — compare with A.9; (2) decode by hand the first 54 bytes of a TCP SYN you capture yourself — done for you at the end of **Task 3** below, check every field; (3) five lines on whether the Saltzer–Reed–Clark argument applies to e-mail delivery.

**✍️ Your answers** (double-click to edit)

- **Homework 3 — the end-to-end argument and e-mail:** 

---
# Part C · Laboratory 2 (Linux) — packet analysis with tcpdump, tshark and Scapy
Colab has no graphical desktop, so **Wireshark's three panes are replaced by code**: `tcpdump` captures, `tshark` applies Wireshark's own display filters and statistics, and Scapy lets us open every header. Every capture file is saved, and you can download it (last cell of this part) and open it in Wireshark on your own computer.

In [ ]:
from scapy.all import rdpcap, Ether, IP, TCP, UDP, ICMP, DNS, Raw, IPerror, UDPerror, TCPerror
IFACE, GATEWAY, MY_IP = default_route()
WEB_HOST = 'example.org'
print(f'Capturing on {IFACE}; my address {MY_IP}; gateway {GATEWAY}')

def capture(action, out, bpf='', iface=None, pre=1.5, post=1.5):
    """Start tcpdump, run action(), stop tcpdump. Returns the list of packets (Scapy)."""
    cmd = f'{SUDO}tcpdump -i {iface or IFACE} -n -U -s 0 -w {out} {bpf}'
    p = subprocess.Popen(shlex.split(cmd), stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    time.sleep(pre)
    try:
        action()
    except Exception as e:
        print('action failed:', e)
    time.sleep(post)
    p.send_signal(signal.SIGINT)
    try:
        err = p.communicate(timeout=10)[1]
    except subprocess.TimeoutExpired:
        p.kill(); err = p.communicate()[1]
    summary = [l for l in err.splitlines() if 'captured' in l or 'error' in l.lower()]
    print(f'[{out}]', '; '.join(summary) or err.strip()[-200:])
    try:
        return rdpcap(out)
    except Exception as e:
        print('could not read capture:', e); return []

def question(d):
    """First question record of a DNS message (works across Scapy versions)."""
    qd = d.qd
    if not qd: return None
    return qd[0] if isinstance(qd, list) else qd     # a list in Scapy ≥ 2.6, a single record before

def info(p):
    """A one-line description, like Wireshark's Info column."""
    if p.haslayer(DNS):
        d = p[DNS]; qr = question(d)
        q = qr.qname.decode() if qr else '?'
        qt = {1: 'A', 28: 'AAAA', 5: 'CNAME', 65: 'HTTPS'}.get(qr.qtype, qr.qtype) if qr else ''
        return f'DNS {"response" if d.qr else "query"} {qt} {q}'
    if p.haslayer(TCP):
        t = p[TCP]; fl = str(t.flags)
        names = {'S': 'SYN', 'SA': 'SYN-ACK', 'A': 'ACK', 'FA': 'FIN-ACK', 'PA': 'PSH-ACK', 'R': 'RST', 'RA': 'RST-ACK', 'F': 'FIN', 'FPA': 'FIN-PSH-ACK'}
        s = f'TCP {t.sport}→{t.dport} [{names.get(fl, fl)}] len={len(t.payload)}'
        if p.haslayer(Raw):
            load = p[Raw].load
            if load[:4] in (b'GET ', b'POST', b'HEAD'):
                s = 'HTTP ' + load.split(b'\r\n')[0].decode(errors='replace')
            elif load[:5] == b'HTTP/':
                s = 'HTTP ' + load.split(b'\r\n')[0].decode(errors='replace')
            elif load[:1] == b'\x16' and len(load) > 5:
                s = 'TLS ' + {1: 'Client Hello', 2: 'Server Hello'}.get(load[5], 'handshake')
            elif load[:1] == b'\x17':
                s = 'TLS application data (encrypted)'
        return s
    if p.haslayer(ICMP):
        i = p[ICMP]
        return f'ICMP {dict([(0,"Echo Reply"),(3,"Destination Unreachable"),(8,"Echo Request"),(11,"Time Exceeded")]).get(i.type, i.type)} (type {i.type}, code {i.code})'
    if p.haslayer(UDP):
        return f'UDP {p[UDP].sport}→{p[UDP].dport}'
    return p.summary()

def protocol(p):
    for layer, name in [(DNS, 'DNS'), (ICMP, 'ICMP')]:
        if p.haslayer(layer): return name
    if p.haslayer(TCP):
        port = min(p[TCP].sport, p[TCP].dport)
        return {80: 'HTTP/TCP', 443: 'TLS/TCP'}.get(port, 'TCP')
    if p.haslayer(UDP): return 'UDP'
    if p.haslayer(IP): return 'IP'
    return p.lastlayer().name

def packet_list(pkts, flt=lambda p: True, limit=40):
    t0 = pkts[0].time if pkts else 0
    shown = 0
    print(f'{"No.":>4s} {"Time":>8s}  {"Source":15s} {"Destination":15s} {"Protocol":9s} Info')
    for i, p in enumerate(pkts, 1):
        if not flt(p): continue
        src = p[IP].src if p.haslayer(IP) else '?'; dst = p[IP].dst if p.haslayer(IP) else '?'
        print(f'{i:4d} {float(p.time - t0):8.4f}  {src:15s} {dst:15s} {protocol(p):9s} {info(p)}')
        shown += 1
        if shown >= limit: print('...'); break

## Task 1 — A first capture

In [ ]:
first = capture(lambda: sh(f'curl -4 -s http://{WEB_HOST}/ -o /dev/null'), 'first.pcap')
packet_list(first, limit=20)

In [ ]:
print('Total packets:', len(first))
print('By protocol  :', dict(Counter(protocol(p) for p in first)))
server_ips = {p[IP].dst for p in first if p.haslayer(TCP) and p[TCP].dport == 80}
mine = [p for p in first if p.haslayer(DNS) or (p.haslayer(IP) and (p[IP].src in server_ips or p[IP].dst in server_ips))]
print('Belonging to curl (DNS + the web server):', len(mine), '\n')
ids = {id(x) for x in mine}
packet_list(first, lambda p: id(p) in ids)

# "Packet details" pane: the layers of one packet, outermost first
if mine:
    p = next((x for x in mine if x.haslayer(Raw)), mine[0])
    layer, names = p, []
    while layer:
        names.append(layer.name); layer = layer.payload
    print('\nLayers of one packet:', ' / '.join(names))
    p.show()

**✍️ Your answers** (double-click to edit)

- **Q1 — total packets; how many belong to curl; what else does the machine send?:** 
- **Q2 — layers listed and their match to the 5-layer model:** 

## Task 2 — Capture filters and display filters
| | Capture filter | Display filter |
|---|---|---|
| When it acts | before capture: unmatched packets are never recorded | after capture: unmatched packets are only hidden |
| Syntax | BPF, as in tcpdump | Wireshark's field language |
| Examples | `host example.org` · `port 53` · `tcp port 80` · `icmp` | `dns` · `http` · `tcp.port == 80` · `icmp` · `tcp.flags.syn == 1` |

In [ ]:
filtered = capture(lambda: sh(f'curl -4 -s http://{WEB_HOST}/ -o /dev/null', show=False),
                   'filtered.pcap', bpf='port 53 or port 80')
print(f'With the capture filter, only {len(filtered)} packets were recorded.\n')
DISPLAY_FILTERS = ['dns', 'http', 'tcp.port == 80', 'tcp.flags.syn == 1', 'dns.flags.response == 1', f'ip.src == {MY_IP}']
for flt in DISPLAY_FILTERS:
    if have('tshark'):
        out = sh(f"tshark -r filtered.pcap -Y '{flt}' 2>/dev/null", show=False)
        print(f'--- display filter: {flt}   ({len(out.strip().splitlines())} packets)')
        print('\n'.join(out.strip().splitlines()[:6]))

**✍️ Your answers** (double-click to edit)

- **Q3 — display filter for DNS responses only; display filter for packets sent by your machine:** 

## Task 3 — One web request, layer by layer
The request must start with a real DNS lookup. If this VM uses a local caching resolver (`127.0.0.x` in `/etc/resolv.conf`), its query would not appear on the network interface, so the notebook then performs the DNS step explicitly against a public resolver just before the HTTP request.

In [ ]:
ns = re.findall(r'^nameserver\s+(\S+)', open('/etc/resolv.conf').read(), re.M)
local_resolver = not ns or ipaddress.ip_address(ns[0]).is_loopback
print('Resolver(s):', ns, '→', 'local cache, DNS step done explicitly' if local_resolver else 'queries go out on the interface')
if have('resolvectl'):
    sh(f'{SUDO}resolvectl flush-caches', show=False)

def web_request():
    if local_resolver:
        addr = sh(f'dig +short A {WEB_HOST} @8.8.8.8 | tail -1', show=False).strip()
        sh(f'curl -4 -s --resolve {WEB_HOST}:80:{addr} http://{WEB_HOST}/ -o /dev/null', show=False)
    else:
        sh(f'curl -4 -s http://{WEB_HOST}/ -o /dev/null', show=False)

web = capture(web_request, 'web.pcap')
if have('editcap'):
    sh('editcap -F pcapng web.pcap web.pcapng', show=False); print('saved web.pcapng')
web_server = next((p[IP].dst for p in web if p.haslayer(TCP) and p[TCP].dport == 80 and str(p[TCP].flags) == 'S'), None)
dns_server = next((p[IP].dst for p in web if p.haslayer(DNS) and not p[DNS].qr), None)
print('web server:', web_server, '| DNS server:', dns_server, '\n')
conv = lambda p: p.haslayer(DNS) or (p.haslayer(TCP) and web_server in (p[IP].src, p[IP].dst))
packet_list(web, conv)

In [ ]:
# Fill in the table of the lab sheet from the HTTP GET packet
get = next((p for p in web if p.haslayer(Raw) and p[Raw].load.startswith(b'GET ')), None)
if get is None:
    print('No HTTP GET found — was the site redirected to HTTPS? Use http:// exactly.')
else:
    e, i, t = get[Ether], get[IP], get[TCP]
    lines = get[Raw].load.split(b'\r\n')
    host = next((l.decode() for l in lines if l.lower().startswith(b'host:')), '')
    rows = [('Ethernet', 'Source MAC / Destination MAC', f'{e.src} / {e.dst}'),
            ('Ethernet', 'Type', f'0x{e.type:04x} ({ETHERTYPE.get(e.type)})'),
            ('IPv4', 'Source / Destination address', f'{i.src} / {i.dst}'),
            ('IPv4', 'TTL · Protocol · Total length', f'{i.ttl} · {i.proto} ({IPPROTO.get(i.proto)}) · {i.len}'),
            ('TCP', 'Source port / Destination port', f'{t.sport} / {t.dport}'),
            ('TCP', 'Sequence number · Header length', f'{t.seq} · {t.dataofs * 4} bytes'),
            ('HTTP', 'Request line · Host header', f'{lines[0].decode()} · {host}')]
    for layer, field, value in rows:
        print(f'{layer:9s} {field:34s} {value}')

In [ ]:
# Q5 — destination MAC of the GET vs the gateway's MAC
sh(f'ip neigh show {GATEWAY}')
if get is not None:
    gw_mac = re.search(r'lladdr (\S+)', sh(f'ip neigh show {GATEWAY}', show=False))
    print('destination MAC of the GET:', get[Ether].dst, '| gateway MAC:', gw_mac.group(1) if gw_mac else '?')
# Q6 — TTL of the server's reply
reply = next((p for p in web if p.haslayer(TCP) and p[IP].src == web_server), None)
if reply is not None:
    ttl = reply[IP].ttl; start = min(x for x in (64, 128, 255) if x >= ttl)
    print(f"server's TTL {ttl} → probably started at {start} → about {start - ttl} routers on the return path")
# Q7 — header bytes before the first byte of HTTP
if get is not None:
    hdr_bytes = len(get) - len(get[Raw].load)
    print(f'{hdr_bytes} header bytes = Ethernet 14 + IP {get[IP].ihl*4} + TCP {get[TCP].dataofs*4} '
          f'(the TCP header carries {get[TCP].dataofs*4 - 20} bytes of options, e.g. timestamps)')

**✍️ Your answers** (double-click to edit)

- **Q4 — demultiplexing keys; what if the EtherType were 0x86DD?:** 
- **Q5 — destination MAC vs gateway MAC: explain:** 
- **Q6 — TTL and number of routers; why can the two directions differ?:** 
- **Q7 — header bytes before HTTP, vs Lecture 2 §3.1:** 

### Homework: decode your own SYN by hand
The first bytes of the SYN you just sent, decoded with the Seminar's decoder. Check every field against the `tshark -V` output below it.

In [ ]:
syn = next((p for p in web if p.haslayer(TCP) and str(p[TCP].flags) == 'S'), None)
if syn is not None:
    raw = bytes(syn)
    hexdump(raw[:54])
    e_, rest = decode_ethernet(raw); i_, rest = decode_ipv4(rest); t_, _ = decode_tcp(rest)
    print('\nEthernet:', e_); print('IPv4    :', i_); print('TCP     :', t_)
    print(f'\nThe SYN is {len(raw)} bytes: 54 bytes of fixed headers + {t_["options_len"]} bytes of TCP options (MSS, SACK, timestamps, window scale).')
    if have('tshark'):
        idx = web.index(syn) + 1
        sh(f"tshark -r web.pcap -Y 'frame.number == {idx}' -V 2>/dev/null | sed -n '1,80p'")

## Task 4 — The whole conversation
*Follow TCP stream*, *Protocol hierarchy*, *Conversations* and *Flow graph*, without the graphical interface.

In [ ]:
def follow_tcp_stream(pkts, server, port=80):
    parts = {'client→server': {}, 'server→client': {}}
    for p in pkts:
        if p.haslayer(TCP) and p.haslayer(Raw) and server in (p[IP].src, p[IP].dst) and port in (p[TCP].sport, p[TCP].dport):
            d = 'client→server' if p[IP].dst == server else 'server→client'
            parts[d][p[TCP].seq] = p[Raw].load
    for d, segs in parts.items():
        data = b''.join(segs[k] for k in sorted(segs))
        print(f'===== {d} ({len(data)} bytes) =====')
        print(data[:700].decode('latin-1')); print()
follow_tcp_stream(web, web_server)

In [ ]:
if have('tshark'):
    sh("tshark -r web.pcap -q -z io,phs 2>/dev/null")      # Statistics → Protocol Hierarchy
    sh("tshark -r web.pcap -q -z conv,ip 2>/dev/null")     # Statistics → Conversations

In [ ]:
def flow_graph(pkts, client, peers, title):
    """Statistics → Flow Graph: a time–sequence diagram."""
    names = {client: f'client\n{client}'}
    cols = [client] + [p for p in peers if p]
    for pr in peers:
        if pr: names[pr] = ('DNS server' if pr == dns_server else 'web server') + f'\n{pr}'
    x = {h: i for i, h in enumerate(cols)}
    sel = [p for p in pkts if p.haslayer(IP) and p[IP].src in x and p[IP].dst in x and p[IP].src != p[IP].dst]
    if not sel:
        print('nothing to draw'); return
    t0 = sel[0].time
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(sel) + 1.5))
    for h in cols:
        ax.axvline(x[h], color='gray', lw=1)
        ax.text(x[h], -0.8, names[h], ha='center', va='bottom', fontsize=8, fontweight='bold')
    for k, p in enumerate(sel):
        a, z = x[p[IP].src], x[p[IP].dst]
        ax.annotate('', xy=(z, k), xytext=(a, k), arrowprops=dict(arrowstyle='->', color='tab:blue' if a < z else 'tab:green'))
        ax.text((a + z) / 2, k - 0.15, info(p), ha='center', fontsize=7)
        ax.text(-0.45, k, f'{float(p.time - t0)*1e3:7.1f} ms', fontsize=7, va='center')
    ax.set_ylim(len(sel), -1.2); ax.set_xlim(-0.6, len(cols) - 0.6); ax.axis('off'); ax.set_title(title)
    plt.tight_layout(); plt.show()
    print(f'{len(sel)} packets for one small page')

cols_order = [dns_server, web_server] if dns_server else [web_server]
flow_graph(web, MY_IP, cols_order, f'One web request to {WEB_HOST}')

**✍️ Your answers** (double-click to edit)

- **Q8 — label the flow graph: DNS lookup, handshake, request, response, close; how many packets for one page?:** 

## Task 5 — Measuring RTT from the handshake
The time from your SYN to the server's SYN-ACK is one RTT. Wireshark also computes it for every acknowledgement (`tcp.analysis.ack_rtt`).

In [ ]:
syn = next((p for p in web if p.haslayer(TCP) and str(p[TCP].flags) == 'S'), None)
synack = next((p for p in web if p.haslayer(TCP) and str(p[TCP].flags) == 'SA'), None)
if syn is not None and synack is not None:
    print(f'SYN → SYN-ACK: {float(synack.time - syn.time)*1e3:.2f} ms  (handshake RTT)')
if have('tshark'):
    sh("tshark -r web.pcap -Y 'tcp.analysis.ack_rtt' -T fields -e frame.number -e ip.src -e tcp.analysis.ack_rtt 2>/dev/null | head")
out = sh(f'ping -c 5 -W 2 {WEB_HOST}', timeout=20)
if ' 0 received' in out or 'received' not in out:
    print('ICMP blocked or unanswered → TCP-connect RTT instead:'); tcp_rtt(WEB_HOST, 80, 5)

# DNS: time between the query and its response
qs = {p[DNS].id: p.time for p in web if p.haslayer(DNS) and not p[DNS].qr}
for p in web:
    if p.haslayer(DNS) and p[DNS].qr and p[DNS].id in qs:
        print(f'DNS {info(p)}: answered in {float(p.time - qs[p[DNS].id])*1e3:.2f} ms')

In [ ]:
# Q10 — repeat WITHOUT flushing any cache: is there still a DNS lookup?
again = capture(lambda: sh(f'curl -4 -s http://{WEB_HOST}/ -o /dev/null', show=False), 'again.pcap', bpf='port 53 or port 80')
n_dns = sum(1 for p in again if p.haslayer(DNS))
print(f'DNS packets the second time: {n_dns}. ' + ('Without a caching resolver on this host, every program asks again.' if n_dns
      else 'No lookup: the answer came from a local cache, valid for the TTL of the DNS record.'))

**✍️ Your answers** (double-click to edit)

- **Q9 — handshake RTT vs ping; explain any difference:** 
- **Q10 — DNS response time; is there a lookup the second time, and why?:** 

## Task 6 — Traceroute under the microscope
Linux `traceroute` sends UDP probes to high ports (from 33434 up) with TTL = 1, 2, 3, … The routers answer ICMP Time Exceeded and quote the start of the original probe inside their message.

In [ ]:
target_ip = resolve(WEB_HOST)
tr = capture(lambda: sh(f'traceroute -n -q 1 -w 2 -m 6 {WEB_HOST}', timeout=60), 'traceroute.pcap',
             bpf=f'icmp or (udp and dst host {target_ip})' if target_ip else 'icmp or udp')
probes = [p for p in tr if p.haslayer(UDP) and not p.haslayer(ICMP) and p[IP].src == MY_IP and p[UDP].dport >= 33434]
print('Outgoing probes:')
for p in probes:
    print(f'   UDP {p[UDP].sport} → {p[IP].dst}:{p[UDP].dport}   TTL = {p[IP].ttl}')
print('Replies:')
for p in tr:
    if p.haslayer(ICMP) and p[IP].dst == MY_IP:
        quoted = ''
        if p.haslayer(IPerror):
            q = p[IPerror]
            quoted = f'  quotes: {q.src} → {q.dst} TTL={q.ttl}' + (f' UDP dport {p[UDPerror].dport}' if p.haslayer(UDPerror) else '')
        print(f'   from {p[IP].src:15s} {info(p)}{quoted}')
if not any(p.haslayer(ICMP) for p in tr):
    print('   (no ICMP replies: this cloud network filters them — try again from a lab/home machine)')

**✍️ Your answers** (double-click to edit)

- **Q11 — protocol, destination ports and TTL of the probes:** 
- **Q12 — ICMP type and code of the replies; why carry the original probe inside?:** 

## Task 7 — Command-line capture: tcpdump and tshark

In [ ]:
def tcpdump_live(args, action, seconds=8):
    p = subprocess.Popen(f'{SUDO}timeout {seconds} tcpdump {args}', shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    time.sleep(1.5)
    try: action()
    except Exception as e: print('action failed:', e)
    print(p.communicate()[0])

req = lambda: sh(f'curl -4 -s http://{WEB_HOST}/ -o /dev/null', show=False)
print('##### with -n (addresses only)'); tcpdump_live(f"-i {IFACE} -n -c 12 'port 53 or port 80'", req)
print('##### without -n (tcpdump looks up names, which is slower)'); tcpdump_live(f"-i {IFACE} -c 12 'port 80'", req, seconds=15)

In [ ]:
cli = capture(req, 'cli.pcap', bpf=f'host {WEB_HOST}')
if have('tshark'):
    sh('tshark -r cli.pcap 2>/dev/null')
    sh("tshark -r cli.pcap -Y http.request -T fields -e ip.src -e ip.dst -e http.host -e http.request.uri 2>/dev/null")

**✍️ Your answers** (double-click to edit)

- **Q13 — is it the same conversation as in Task 3? What does -n change?:** 

## Task 8 — HTTPS: what remains visible
The HTTP request is encrypted, but the **Client Hello** still carries the server's name in the `server_name` (SNI) extension. Below, the Client Hello is parsed byte by byte.

In [ ]:
def client_hello_sni(data):
    """Return the server_name from a TLS Client Hello (bytes starting at the TLS record)."""
    try:
        if data[0] != 0x16 or data[5] != 0x01: return None
        p = 5 + 4 + 2 + 32                         # record header, handshake header, version, random
        p += 1 + data[p]                           # session id
        p += 2 + int.from_bytes(data[p:p+2], 'big')  # cipher suites
        p += 1 + data[p]                           # compression methods
        end = p + 2 + int.from_bytes(data[p:p+2], 'big'); p += 2
        while p + 4 <= min(end, len(data)):
            etype, elen = int.from_bytes(data[p:p+2], 'big'), int.from_bytes(data[p+2:p+4], 'big'); p += 4
            if etype == 0:                         # server_name
                n = int.from_bytes(data[p+3:p+5], 'big'); return data[p+5:p+5+n].decode()
            p += elen
    except IndexError:
        return None

tls = capture(lambda: sh(f'curl -4 -s https://{WEB_HOST}/ -o /dev/null', show=False), 'https.pcap', bpf='port 53 or tcp port 443')

# Find our connection: a flow that STARTS (SYN) during the capture and whose Client Hello names WEB_HOST
flows = {}
for p in tls:
    if p.haslayer(TCP) and p.haslayer(IP) and p[IP].src == MY_IP and p[TCP].dport == 443:
        key = (p[IP].dst, p[TCP].sport)
        if str(p[TCP].flags) == 'S': flows.setdefault(key, {})
        if key in flows and p.haslayer(Raw): flows[key][p[TCP].seq] = p[Raw].load
srv443, hello, my_port = None, b'', None
for (dst, sport), segs in flows.items():
    data = b''.join(segs[k] for k in sorted(segs))[:4000]
    if client_hello_sni(data) == WEB_HOST or srv443 is None:
        srv443, hello, my_port = dst, data, sport
conn = lambda p: p.haslayer(DNS) or (p.haslayer(TCP) and srv443 in (p[IP].src, p[IP].dst))
packet_list(tls, conn, limit=25)
print('\nSNI found by our parser:', client_hello_sni(hello))
if have('tshark'):
    sh("tshark -r https.pcap -Y 'tls.handshake.type == 1' -T fields -e ip.dst -e tls.handshake.extensions_server_name 2>/dev/null")

In [ ]:
# Q15 — what an observer on the network still learns
names = sorted({question(p[DNS]).qname.decode() for p in tls if p.haslayer(DNS) and question(p[DNS])})
flows = [p for p in tls if p.haslayer(TCP) and srv443 in (p[IP].src, p[IP].dst) and my_port in (p[TCP].sport, p[TCP].dport)]
up = sum(len(p) for p in flows if p[IP].dst == srv443); down = sum(len(p) for p in flows if p[IP].src == srv443)
print('Observer sees:')
print(' • the name you looked up (DNS, in clear):', names)
print(' • the server name in the Client Hello (SNI):', client_hello_sni(hello))
print(' • addresses and ports:', MY_IP, '↔', f'{srv443}:443')
print(f' • sizes: {up} bytes up, {down} bytes down, {len(flows)} packets')
if flows: print(f' • timing: the visit lasted {float(flows[-1].time - flows[0].time)*1e3:.0f} ms')
print('Observer does NOT see: the URL path, the HTTP headers, cookies, the page content.')

**✍️ Your answers** (double-click to edit)

- **Q14 — can you read the request? What does server_name contain?:** 
- **Q15 — what an observer can still learn:** 
- **Q16 — which layers TLS encrypts and which travel in clear; relation to the end-to-end argument:** 

### Download the capture files (to open in Wireshark on your own computer)

In [ ]:
files_to_get = [f for f in ['web.pcapng', 'web.pcap', 'traceroute.pcap', 'https.pcap', 'cli.pcap'] if os.path.exists(f)]
print('Available:', files_to_get)
if IN_COLAB:
    from google.colab import files
    for f in files_to_get:
        if f != 'web.pcap' or 'web.pcapng' not in files_to_get:
            files.download(f)

---
# Part D · Laboratory 2 (Windows) — Wireshark and pktmon, seen from here
Wireshark is the same program on Windows; what differs is the capture driver (**Npcap**), the built-in command-line capture tool (**pktmon**, administrator only), and **`tracert`**, which sends ICMP Echo Requests instead of UDP. The last difference we can reproduce exactly: `traceroute -I` probes like `tracert`.

## Task 6 (Windows) — `tracert` under the microscope
`tracert -d -h 6 -w 1000 example.org` sends **3 ICMP Echo Requests per hop**; the destination answers with an Echo Reply.

In [ ]:
tw = capture(lambda: sh(f'{SUDO}traceroute -I -n -q 3 -w 1 -m 6 {WEB_HOST}', timeout=60), 'tracert_style.pcap', bpf='icmp')
echo = [p for p in tw if p.haslayer(ICMP) and p[ICMP].type == 8 and p[IP].src == MY_IP]
print(f'{len(echo)} ICMP Echo Requests sent; TTLs:', Counter(p[IP].ttl for p in echo))
print('Reply types:', Counter(info(p) for p in tw if p.haslayer(ICMP) and p[IP].dst == MY_IP))
print('\nCompare with Linux traceroute in Task 6: UDP to ports ≥ 33434 → the destination answers Port Unreachable.')

**✍️ Your answers** (double-click to edit)

- **Q11 (Windows) — kind of packets, TTL per hop, probes per hop; difference from Linux traceroute:** 
- **Q12 (Windows) — ICMP type of the replies and what is quoted inside:** 

## Linux and Windows side by side

In [ ]:
SIDE = [('Capture driver', 'kernel (libpcap)', 'Npcap'),
        ('Graphical capture', 'Wireshark', 'Wireshark'),
        ('Built-in command-line capture', 'tcpdump', 'pktmon (admin)'),
        ('Command-line analysis', 'tshark', 'tshark.exe'),
        ('Flush the DNS cache', 'resolvectl flush-caches', 'ipconfig /flushdns'),
        ('Traceroute probes', 'UDP to high ports (default)', 'ICMP Echo Request'),
        ('Typical initial TTL', '64', '128')]
print(f'{"Purpose":30s} {"Linux":28s} Windows')
for row in SIDE: print(f'{row[0]:30s} {row[1]:28s} {row[2]}')
print('\nInitial TTL of this Linux VM:', open('/proc/sys/net/ipv4/ip_default_ttl').read().strip())

### pktmon, for your Windows machine
```powershell
# (admin)
pktmon start --capture --comp nics --pkt-size 0 -f cli.etl
curl.exe -s http://example.org/ -o NUL
pktmon stop
pktmon etl2pcap cli.etl -o cli.pcapng
& "C:\Program Files\Wireshark\tshark.exe" -r cli.pcapng -Y http.request -T fields -e ip.src -e ip.dst -e http.host -e http.request.uri
```
**Q13 (Windows)** — `pktmon` sees each packet at several points inside the network stack, so some packets appear more than once. Compare its output with `cli.pcap` from Task 7, which you can download above.

**Wi-Fi note.** On a Wi-Fi adapter, both Windows and Linux (in normal mode) hand Wireshark frames that look like Ethernet; the real 802.11 headers appear only in monitor mode.

**✍️ Your answers** (double-click to edit)

- **Q13 (Windows) — duplicates in pktmon; same conversation as Task 3?:** 

---
### Next week
Lecture 3 opens the application layer with the Web and HTTP. In Laboratory 3 you write your first network programs: a TCP client and server using sockets.